# 🔍 Pay Equity Lens — Deep Dive Insights
Building on EDA findings — this notebook quantifies the pay gap rigorously and trains our salary benchmarking model.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error

df = pd.read_csv('../data/processed/hr_clean.csv')
print(f"Loaded: {df.shape}")

## 1. Statistical Pay Gap Analysis

In [ ]:
# Pay gap by gender controlling for job level
gap_table = df.groupby(['JobLevelLabel', 'Gender'])['MonthlyIncome'].mean().unstack().round(0)
gap_table['Gap ($)'] = (gap_table['Male'] - gap_table['Female']).round(0)
gap_table['Gap (%)'] = ((gap_table['Gap ($)'] / gap_table['Male']) * 100).round(1)
print("Pay Gap by Job Level:")
print(gap_table)

print("\nPay Gap by Department:")
dept_gap = df.groupby(['Department', 'Gender'])['MonthlyIncome'].mean().unstack().round(0)
dept_gap['Gap ($)'] = (dept_gap['Male'] - dept_gap['Female']).round(0)
dept_gap['Gap (%)'] = ((dept_gap['Gap ($)'] / dept_gap['Male']) * 100).round(1)
print(dept_gap)

## 2. Salary Benchmarking Model

In [ ]:
# Encode features
le = LabelEncoder()
df_model = df.copy()
cat_cols = ['Department', 'Gender', 'JobRole', 'MaritalStatus', 'BusinessTravel', 'OverTime', 'EducationField', 'Attrition']
for col in cat_cols:
    df_model[col + '_enc'] = le.fit_transform(df_model[col])

features = ['Age', 'Education', 'JobLevel', 'JobSatisfaction', 'TotalWorkingYears',
            'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'TrainingTimesLastYear', 
            'PerformanceRating', 'Department_enc', 'Gender_enc', 'JobRole_enc', 
            'MaritalStatus_enc', 'BusinessTravel_enc', 'OverTime_enc']

X = df_model[features]
y = df_model['MonthlyIncome']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=10),
    'Random Forest': RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=200, max_depth=5, random_state=42),
}

print(f"{'Model':<25} {'MAE':>10} {'RMSE':>10} {'R²':>8}")
print("-" * 55)
results = {}
for name, model in models.items():
    use_sc = 'Regression' in name
    model.fit(X_train_sc if use_sc else X_train, y_train)
    preds = model.predict(X_test_sc if use_sc else X_test)
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)
    results[name] = {'MAE': mae, 'RMSE': rmse, 'R2': r2, 'preds': preds}
    print(f"{name:<25} ${mae:>8,.0f} ${rmse:>8,.0f} {r2:>7.4f}")

## 3. Pay Equity Flagging

In [ ]:
rf_model = models['Random Forest']
df_model['PredictedFairSalary'] = rf_model.predict(X).astype(int)
df_model['SalaryGap'] = df_model['MonthlyIncome'] - df_model['PredictedFairSalary']
df_model['PayStatus'] = 'Fair'
df_model.loc[df_model['SalaryGap'] < -500, 'PayStatus'] = 'Underpaid'
df_model.loc[df_model['SalaryGap'] > 500, 'PayStatus'] = 'Overpaid'

print("Pay Status Distribution:")
print(df_model['PayStatus'].value_counts())
print(f"\n% Underpaid by Gender:")
print((df_model.groupby('Gender')['PayStatus'].apply(lambda x: (x=='Underpaid').mean()*100)).round(1))
print(f"\n% Underpaid by Department:")
print((df_model.groupby('Department')['PayStatus'].apply(lambda x: (x=='Underpaid').mean()*100)).round(1))

df_model.to_csv('../data/processed/hr_with_predictions.csv', index=False)
print("\nPredictions saved.")

## 4. Model Visualization

In [ ]:
plt.rcParams.update({
    'figure.facecolor': '#0f1117', 'axes.facecolor': '#1a1d2e',
    'axes.labelcolor': 'white', 'xtick.color': 'white', 'ytick.color': 'white',
    'text.color': 'white', 'axes.titlecolor': 'white', 'axes.edgecolor': '#333355', 'grid.color': '#333355',
})

feat_labels = {
    'JobLevel': 'Job Level', 'TotalWorkingYears': 'Total Working Years',
    'YearsAtCompany': 'Years at Company', 'Age': 'Age',
    'YearsInCurrentRole': 'Years in Role', 'JobRole_enc': 'Job Role',
    'Department_enc': 'Department', 'Education': 'Education',
    'Gender_enc': 'Gender', 'PerformanceRating': 'Performance Rating',
    'BusinessTravel_enc': 'Business Travel', 'JobSatisfaction': 'Job Satisfaction',
    'MaritalStatus_enc': 'Marital Status', 'OverTime_enc': 'Overtime',
    'WorkLifeBalance': 'Work-Life Balance', 'TrainingTimesLastYear': 'Training Times',
}

fi = pd.Series(rf_model.feature_importances_, index=features).sort_values(ascending=True)
fi.index = [feat_labels.get(i, i) for i in fi.index]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.patch.set_facecolor('#0f1117')

bar_cols = ['#7B5EA7' if v > fi.mean() else '#54A0E0' for v in fi.values]
axes[0].barh(fi.index, fi.values, color=bar_cols, edgecolor='#0f1117')
axes[0].set_title('Feature Importance\n(Salary Predictor)', fontweight='bold')
axes[0].set_xlabel('Importance Score')
axes[0].grid(True, alpha=0.3, axis='x')

rf_res = results['Random Forest']
axes[1].scatter(y_test, rf_res['preds'], alpha=0.4, s=15, c='#7B5EA7')
mn, mx = y_test.min(), y_test.max()
axes[1].plot([mn, mx], [mn, mx], 'r--', lw=2, label='Perfect Prediction')
axes[1].set_title(f'Actual vs Predicted\nR² = {rf_res["R2"]:.4f}', fontweight='bold')
axes[1].set_xlabel('Actual Monthly Income ($)')
axes[1].set_ylabel('Predicted Monthly Income ($)')
axes[1].legend(facecolor='#1a1d2e', labelcolor='white')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../reports/fig5_model_results.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()
print("Chart saved.")